## Pipeline de métricas de diversidad B immuneSIM

Este pipeline se utiliza para calcular y analizar 13 métricas de diversidad a partir de los repertorios simulados. Su objetivo es caracterizar la estructura clonal bajo distintos escenarios y profundidades de secuenciación, permitiendo comparaciones sistemáticas de la diversidad entre condiciones.

Las métricas obtenidas incluyen índices de riqueza, diversidad, equidad y dominancia, y son utilizadas como base para los análisis posteriores del estudio.

In [362]:
# Paquetes y librerías
library(readr)
library(dplyr)
library(ggplot2)
library(viridisLite)
library(here)  # para rutas relativas
# install.packages("alakazam")
library(alakazam)
library(ineq)
library(vegan)
library(tidyr)
library(tibble)

## Carga de datos y obtención de abundancias clonales

En esta sección se carga un archivo `.tsv` correspondiente a un repertorio simulado generado con **immuneSIM**.

Posteriormente:

- Se extraen las abundancias clonales desde la columna `counts`, generada directamente por el simulador.
- Estas abundancias corresponden al número de secuencias asociadas a cada clon y constituyen el vector de entrada para el cálculo de las métricas de diversidad.

La tabla (`clone_counts`) y su columna `counts` serán utilizadas posteriormente para el cálculo de las métricas de diversidad clonotípica.

In [363]:
gt_file <- "/Users/catg/Desktop/SOFIAC/Gitsofia/tesisbioinf-sofia/results/immunesim_ground_truth/gt_B_12800.tsv"

gt <- read_tsv(gt_file)

clone_counts <- gt %>%
  select(junction, counts)

Rows: 12800 Columns: 7
-- Column specification --------------------------------------------------------
Delimiter: "\t"
chr (5): sequence, v_call, d_call, j_call, junction
dbl (2): counts, freqs

i Use `spec()` to retrieve the full column specification for this data.
i Specify the column types or set `show_col_types = FALSE` to quiet this message.


## Números de Hill como métricas unificadas de diversidad

Los números de Hill representan un marco unificado para el cálculo de métricas de diversidad, permitiendo expresar distintas medidas en una misma escala de **número efectivo de clones**.

Este enfoque integra métricas clásicas como riqueza, Shannon y Simpson dentro de un mismo sistema, facilitando la comparación directa entre repertorios clonales.

A partir de la tabla de números de Hill (`rep_hill_numbers`), se extraen los valores de diversidad correspondientes a distintos órdenes (q), filtrando la tabla según cada valor y obteniendo la diversidad asociada (`d`).

### Métricas representadas por los números de Hill

* **q = 0 → Richness (riqueza clonal)**
  Representa el número total de clones únicos presentes en el repertorio, sin considerar sus abundancias.

* **q = 1 → Shannon (número efectivo de clones)**
  Corresponde a la transformación exponencial del índice de Shannon, incorporando la frecuencia relativa de los clones.

* **q = 2 → Simpson (dominancia clonal)**
  Da mayor peso a los clones más abundantes, permitiendo evaluar la dominancia dentro del repertorio.

* **q = 3 y q = 4 → órdenes superiores de Hill**
  Incrementan progresivamente la influencia de los clones dominantes, capturando estructuras de dominancia más marcadas.

Los valores obtenidos para cada orden de diversidad se utilizan posteriormente para comparar la diversidad clonotípica entre repertorios simulados y distintas condiciones experimentales.


In [364]:

hill_numbers <- function(clone_counts){

    q_values <- 0:4

    diversity_values <- sapply(q_values, function(q) {
        calcDiversity(clone_counts$counts, q)
    })

    hill_table <- data.frame(
        q = q_values,
        d = diversity_values
    )

    return(hill_table)
}

# Ejecutar
rep_hill_numbers <- hill_numbers(clone_counts)

rep_hill_numbers

q,d
<int>,<dbl>
0,12800.000000
1,1.971076
2,1.420308
3,1.316594
4,1.278001


In [365]:
richness <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 0) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

richness(rep_hill_numbers)

[1] 12800

In [366]:
 d1 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 1) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

d1(rep_hill_numbers)

[1] 1.971076

In [367]:
shannon <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 1) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    
    return(log(metric_value))
} 

shannon(rep_hill_numbers)

[1] 0.6785797

In [368]:
 d2 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 2) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

d2(rep_hill_numbers)

[1] 1.420308

In [369]:
simpson <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 2) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    
    return(1/(metric_value))
} 

simpson(rep_hill_numbers)

[1] 0.7040725

In [370]:
 d3 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 3) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

d3(rep_hill_numbers)

[1] 1.316594

In [371]:
d4 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 4) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

d4(rep_hill_numbers)

[1] 1.278001

## Cálculo de estimadores de riqueza Chao1 y ACE

Se utilizan estimadores no paramétricos de riqueza clonal para evaluar la diversidad total de los repertorios simulados generados con **immuneSIM**. Este análisis se realiza mediante la función `estimateR()` del paquete **vegan**.

Los estimadores **Chao1** y **ACE** permiten aproximar la riqueza clonal real del repertorio considerando la presencia de clones raros, los cuales pueden estar subrepresentados en la muestra.

En este caso, las abundancias clonales (`counts`) provienen directamente del simulador, representando el número de células asociadas a cada clon. Estas se agrupan por `sample_id` para cada repertorio simulado.

A partir de esta distribución se calculan los siguientes estimadores:

* **Chao1**: estima el número total de clones esperados en el repertorio, incorporando la contribución de clones observados una o pocas veces.
* **ACE (Abundance-based Coverage Estimator)**: estima la riqueza clonal considerando la abundancia de clones poco frecuentes y el nivel de cobertura del muestreo.

Los valores obtenidos se almacenan en una tabla (`metricas_chao1ace`) para su posterior análisis y comparación entre repertorios.


In [372]:
abundances <- clone_counts$counts

metricas_chao1ace <- data.frame(
  chao1 = estimateR(abundances)["S.chao1"],
  ace   = estimateR(abundances)["S.ACE"]
)

metricas_chao1ace

,chao1,ace
,<dbl>,<dbl>
S.chao1,13548.01,13838.66


In [373]:
chao1 <- function(clones_df){

  metricas_chao1 <- as.numeric(
    vegan::estimateR(clones_df$counts)["S.chao1"]
  )

  return(metricas_chao1)
}
rep_chao1 <- chao1(clone_counts)
print(rep_chao1)



[1] 13548.01


In [374]:
ace <- function(clones_df){

  metricas_ace <- as.numeric(
    vegan::estimateR(clones_df$counts)["S.ACE"]
  )

  return(metricas_ace)
}
rep_ace <- ace(clone_counts)
print(rep_ace)

[1] 13838.66


## Cálculo del índice de Gini (desigualdad clonal)

Se calcula el índice de Gini para evaluar el grado de desigualdad en la distribución de abundancias clonales dentro de los repertorios simulados generados con **immuneSIM**. Este análisis se realiza utilizando la función `ineq()` del paquete **ineq**.

El índice de Gini permite cuantificar la dominancia clonal, es decir, qué tan concentrada se encuentra la abundancia de secuencias en pocos clones.

Para su cálculo, se aplica una función personalizada (`calc_gini`) sobre las abundancias clonales (`counts`) provenientes directamente del simulador. Posteriormente, los datos se agrupan por `sample_id` (o identificador del repertorio simulado) y se obtiene un valor de Gini por muestra.

### Interpretación del índice de Gini

* **Gini ≈ 0** → distribución uniforme de clones (abundancias similares entre clones).
* **Gini ≈ 1** → alta desigualdad clonal (pocos clones dominan el repertorio).

Los valores obtenidos se almacenan en una tabla (`gini_result`) para su posterior análisis comparativo entre repertorios clonales.


In [375]:
library(ineq)

calc_gini <- function(df) {
  ineq::ineq(df$counts, type = "Gini")
}

gini_result <- data.frame(
  gini = calc_gini(clone_counts)
)

print(gini_result)

       gini
1 0.9998643


In [376]:
gini <- function(df){
  ineq::ineq(df$counts, type = "Gini")
}

gini(clone_counts)

[1] 0.9998643

## Cálculo del índice de uniformidad de Pielou (evenness)

Se calcula el índice de uniformidad de **Pielou** para evaluar qué tan equitativa es la distribución de abundancias clonales dentro de los repertorios simulados generados con **immuneSIM**. Este análisis se realiza utilizando funciones del paquete **vegan**.

Para su estimación, se utiliza una función personalizada (`calc_pielou`) que combina el cálculo del índice de Shannon (`diversity()`) y el número total de clones (`specnumber()`), a partir de las abundancias clonales (`counts`) provenientes directamente del simulador.

El índice de Pielou se define como:

**J = H / log(S)**

donde:

* **H** corresponde al índice de Shannon.
* **S** corresponde al número total de clones (richness).
* **J** representa la uniformidad en la distribución clonal.

Posteriormente, los datos se agrupan por `sample_id` (o identificador del repertorio simulado) y se obtiene un valor de Pielou para cada muestra.

### Interpretación del índice de Pielou

* **J ≈ 1** → distribución uniforme de clones (abundancias similares entre clones).
* **J ≈ 0** → baja uniformidad (presencia de clones dominantes).

Los valores obtenidos se almacenan en una tabla (`pielou_result`) para su posterior análisis comparativo entre repertorios clonales.


In [377]:
calc_pielou <- function(df) {

  abund <- df$counts
  H <- vegan::diversity(abund, index = "shannon")
  S <- vegan::specnumber(abund)

  H / log(S)
}

calc_pielou(clone_counts)

[1] 0.0717445

## Cálculo del índice de Basharin (corrección de Shannon)

Se calcula el índice de **Basharin**, una variante corregida del índice de Shannon que incorpora un término de ajuste para reducir el sesgo asociado a tamaños de muestra finitos.

Para su estimación, se utiliza una función personalizada (`calc_basharin`) que calcula el índice de Shannon (`diversity()`) y el número total de clones (`specnumber()`), a partir de las abundancias clonales (`counts`) provenientes directamente del simulador.

El índice de Basharin se define como:

**B = H + (S − 1) / (2N)**

donde:

* **H** corresponde al índice de Shannon.
* **S** corresponde al número total de clones (richness).
* **N** corresponde al número total de secuencias.
* **B** corresponde al índice de Basharin corregido.

Este término adicional permite ajustar la estimación de diversidad en escenarios con tamaños de muestra limitados, reduciendo el sesgo del índice de Shannon.

Los cálculos se realizan agrupando los datos por `sample_id` (o identificador del repertorio simulado), obteniendo un valor de Basharin por cada repertorio analizado.

Los valores obtenidos se almacenan en la tabla (`basharin_result`) para su posterior análisis comparativo entre repertorios clonales.


In [378]:
calc_basharin <- function(df) {

  abund <- df$counts

  N <- sum(abund)
  S <- vegan::specnumber(abund)

  if (N == 0 || S <= 1) return(0)

  H <- vegan::diversity(abund, index = "shannon")

  basharin <- H + (S - 1) / (2 * N)

  return(basharin)
}
calc_basharin(clone_counts)

[1] 0.6785021

## Cálculo del índice D50 (dominancia clonal)

Se calcula el índice **D50**, una métrica utilizada para evaluar la dominancia clonal dentro de los repertorios simulados generados con **immuneSIM**.

El índice D50 corresponde al número mínimo de clones más abundantes necesarios para alcanzar el 50% del total de secuencias del repertorio.

Para su cálculo, las abundancias clonales (`counts`) provenientes del simulador se ordenan de mayor a menor, y se calcula la suma acumulativa de las abundancias. El valor de D50 se obtiene como el número de clones requeridos para alcanzar el 50% del total de secuencias.

### Interpretación del índice D50

* **D50 bajo** → alta dominancia clonal (pocos clones representan gran parte del repertorio).
* **D50 alto** → mayor diversidad clonal (se requieren más clones para alcanzar el 50%).

Los valores obtenidos se almacenan en la tabla (`d50_result`) para su posterior análisis comparativo entre repertorios clonales.


In [379]:
d50_fun <- function(counts) {

  counts <- sort(counts, decreasing = TRUE)
  total <- sum(counts)
  cum <- cumsum(counts)

  which(cum >= 0.5 * total)[1]
}
d50_val <- d50_fun(clone_counts$counts)
d50_val

[1] 1

## Integración de métricas de diversidad

Se combinan múltiples métricas de diversidad clonotípica 
(números de Hill, estimadores de riqueza y métricas de 
dominancia y uniformidad) en una única estructura 
(`metricas_diversidad`) para facilitar su análisis posterior.

In [380]:
metricas_diversidad <- c(

  # --- Identidad Hill numbers ---
  richness = richness(rep_hill_numbers),
  d1 = d1(rep_hill_numbers),
  shannon = vegan::diversity(clone_counts$counts, "shannon"),
  d2 = d2(rep_hill_numbers),
  simpson = vegan::diversity(clone_counts$counts, "simpson"),
  d3 = d3(rep_hill_numbers),
  d4 = d4(rep_hill_numbers),

  # --- Abundancia / riqueza estimada ---
  chao1 = as.numeric(vegan::estimateR(clone_counts$counts)["S.chao1"]),
  ace   = as.numeric(vegan::estimateR(clone_counts$counts)["S.ACE"]),

  # --- Desigualdad / equidad ---
  gini = ineq::ineq(clone_counts$counts, type = "Gini"),

  pielou = {
    H <- vegan::diversity(clone_counts$counts, "shannon")
    S <- vegan::specnumber(clone_counts$counts)
    H / log(S)
  },

  basharin = {
    abund <- clone_counts$counts
    N <- sum(abund)
    S <- vegan::specnumber(abund)
    H <- vegan::diversity(abund, "shannon")
    H + (S - 1) / (2 * N)
  },

  d50 = {
    counts <- sort(clone_counts$counts, decreasing = TRUE)
    which(cumsum(counts) >= 0.5 * sum(counts))[1]
  }
)

## Construcción de la tabla final de diversidad

Las métricas de diversidad calculadas se integran en una 
tabla (`tabla_diversidad`) y se añadieron variables 
descriptivas del repertorio analizado.

Se incorporan los siguientes metadatos:

- `sample_id`: identificador del repertorio  
- `condition`: condición o escenario simulado  
- `size`: tamaño del repertorio (número de secuencias)

Esta tabla final permite organizar las métricas y facilitar 
su comparación entre distintos escenarios y tamaños 
de repertorios simulados.

In [381]:
tabla_diversidad <- as.data.frame(t(metricas_diversidad))

tabla_diversidad$sample_id <- "B12800seq"
tabla_diversidad$condition <- "B"
tabla_diversidad$size <-12800

tabla_diversidad <- tabla_diversidad %>%
  dplyr::select(sample_id, condition, size, dplyr::everything())

In [382]:
readr::write_tsv(
  tabla_diversidad,
  "/Users/catg/Desktop/SOFIAC/Gitsofia/tesisbioinf-sofia/results/diversity_metrics/B/diversity_B_12800seqs.tsv"
)